# Semigroups and Monoids in Cats

## Remmember Semigroup

We characterize Semigroups as having:

 - non-empty set with an associative binary operation 

 - for all `x`, `y`, and `z` in the semigroup, and the binary operation `*`:

```
x * (y * z) = (x * y) * z
```

The signature of a Semigroup in Cats is something like:

```
trait Semigroup[A] :
  def combine(x: A, y: A): A 
```

where the `combine` method is the binary operation.

In [ ]:
import $ivy.`org.typelevel::cats-core:2.13.0`

Now, using a `Semigroup` in Cats allow using the `combine` operator:

In [ ]:
import cats.Semigroup

Semigroup[Int].combine(34,23)

A bit of syntax flavor in Cats makes it simpler:

In [ ]:
import cats.syntax.semigroup._

3.combine(4)

In [ ]:
Option(4) combine (Option(5))

We can also use it for functions, as long as the operator is defined:

In [ ]:
val f = ((i:Int)=>i+2) combine ((i:Int)=>i*4) 

In [ ]:
f(4)

In a Map the `combine` deal with keys and values for you:

In [ ]:
val map1 = Map("GVA"->34,"VD"->12)
val map2 = Map("VS"->45,"VD"->15)

map1 combine map2

Using Semigroup syntax addins, we can use the `|+|` symbol to represent the `combine` method:

In [ ]:
import cats.syntax.semigroup._

1 |+| 2

We can use `given` to define custom behaviors for the `combine` or `|+|` operator:

In [ ]:
given Semigroup[Boolean] with
  def combine(a:Boolean,b:Boolean) = a && b

true |+| false
true |+| (true |+| false)
(true |+| true) |+| false


Finally, we can also apply it to Tuples:

In [ ]:
val grades1stYear = (2d,2.5,1d)

val grades2ndYear = (3d,1.5,2.5)

grades1stYear |+| grades2ndYear


## Monoids

**Reminder**: semigroup is a set with an associative operation on it. 

**Monoid:**
A *semigroup* with an *identity* element: 

- The identity element `e` in the semigroup such that: 

```
   e * x = x * e = x
````

for all elements `x` in the semigroup

The signature of a Monoid in Cats looks like:


```
trait Monoid[A] extends Semigroup[A] :
	def empty: A
```

Now we can actually use the Monoid. 

As expected it works just like the Semigroup, but now adding the `empty` attribute representing the *identity*:

In [ ]:
import cats.Monoid

Monoid.combine("hey","you")

In [ ]:
Monoid.combine(23,21)

In [ ]:
Monoid.combine(-23.4,-35.4)

In [ ]:
Monoid[Int].empty

Some additional methods are added as well, as `combineAll` that operates `combine` over collections:

In [ ]:
Monoid.combineAll(List(3,4,5,6))

This has immediate consequences on Types like `Option`, making it simple to operate with them:

In [ ]:
val exam1=Option(3)
val exam2=Option(5)
Monoid.combine(exam1,exam2)

Now we can use the `Monoid` for our own classes, like this `Grade` case class:

In [ ]:
case class Grade(value:Double)

Suffices to use a `given`, and defining the `combine` and `empty` methods:

In [ ]:
given Monoid[Grade] with
  def combine(a:Grade,b:Grade) = Grade((a.value+b.value)/2)
  def empty = Grade(0)

Then using the `combine` operator is straightforward:

In [ ]:
Grade(3) |+| Grade(5)

In [ ]:
Grade(3) |+| (Grade(2) |+| Grade(5))

(Grade(3) |+| Grade(2)) |+| Grade(5)

Notice that careful implementation is important to guarantee that the `combine` method is really associative.

In [ ]:
given Monoid[Grade] with
  def combine(a:Grade,b:Grade) =
     if a.value > b.value then a else b 
  def empty = Grade(0)

In [ ]:
Grade(3) |+| (Grade(2) |+| Grade(5))

(Grade(3) |+| Grade(2)) |+| Grade(5)

As we have seen, `Monoids` can be naturally used with fold oerations, where you can see both the `combine` and `empty` methods:

In [ ]:
val grades = List(4,5.5,3.0,4,2,5).map(Grade(_))

grades.foldRight(Monoid[Grade].empty)(_ |+| _)

Which is transparently done through `combineAll`:

In [ ]:
Monoid[Grade].combineAll(grades)

Again, with types like `Option` the monoid operators help making sense out of it and avoiding messing with them:

In [ ]:
val opGrade1 = Option(Grade(5))
val opGrade3 = None
val opGrade2 = Option(Grade(3))

opGrade1 |+| opGrade2 |+| opGrade3

We can see it similarly with combinations of collections, like `Map` or `List`:

In [ ]:
Monoid[Map[String, Int]].combineAll(List(Map("a" -> 1, "b" -> 2), 
                                         Map("a" -> 3)))

In [ ]:
Semigroup[List[Int]].combine(List(1, 2, 3), List(4, 5, 6))


In [ ]:
Semigroup[Int => Int].combine(_ + 1, _ * 10).apply(6)


Notice how the result of combine can be another `Monoid` or `Semigroup`:

In [ ]:
val aMap = Map("foo" -> Map("bar" -> 5))
val anotherMap = Map("foo" -> Map("bar" -> 6))
val combinedMap = 
	Semigroup[Map[String, Map[String, Int]]].combine(aMap,anotherMap)

combinedMap.get("foo")
